# Statistical Tests

To determine if the patterns observed during EDA occured by chance or not, we check for statistical significane for the following four questions:

1. Did the shock significantly widen spreads?
2. Was the widening asymmetric across sectors?
3. Which option type spread widened more during the shock?
4. Did spreads fully recover?



In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import mannwhitneyu, kruskal
from itertools import combinations


PROJECT_ROOT = Path().resolve().parent
COMBINED_CSV = PROJECT_ROOT / "data" / "combined" / "combined_all.csv"

TICKERS      = ["AAPL", "NVDA", "AMZN", "PG", "CAT"]
COMMON_START = pd.Timestamp("2025-03-24")
ALPHA        = 0.05   # significance threshold

df = pd.read_csv(
    COMBINED_CSV,
    parse_dates=["collection_date", "expiration"],
    dtype={"ticker": str, "side": str, "moneyness_cat": str},
)
df["is_illiquid"] = df["is_illiquid"].astype(bool)

liquid = df[~df["is_illiquid"]].copy()
atm    = liquid[liquid["moneyness_cat"] == "ATM"].copy()
cross  = atm[atm["collection_date"] >= COMMON_START].copy()

## Statistical Tests
Option bid-ask spreads are right-skewed and this violates normality assumptions. We have to use the **Mann-Whitney U Test** over **t-Test**.

The Mann-Whitney U Test works on rankings rather than raw values, so the skew doesn't matter.

We use the **rank-biserial correlation** `r` as our effect size. It ranges from -1 to +1:
- `r ≈ 0` → no difference between groups
- `r > 0` → the second group tends to be larger
- `r < 0` → the first group tends to be larger

Magnitude thresholds (standard convention):
- `|r| < 0.1` → negligible
- `0.1 ≤ |r| < 0.3` → small
- `0.3 ≤ |r| < 0.5` → medium
- `|r| ≥ 0.5` → large

In [3]:
#Helper Functions

def rank_biserial_r(u_stat, n1, n2):
    return 1 - (2 * u_stat) / (n1 * n2)


def interpret_effect(r):
    a = abs(r)
    if a < 0.1:   return "negligible"
    elif a < 0.3: return "small"
    elif a < 0.5: return "medium"
    else:         return "large"


def significance_label(p, alpha=ALPHA):
    if p < 0.001:  return "***Extremely Significant  (p<0.001)" 
    elif p < 0.01: return "**Very Significant  (p<0.01)"
    elif p < 0.05: return "*Significant  (p<0.05)"
    else:          return "Not Significant  (p≥0.05)"


---
## Question 1: Did the shock significantly widen spreads?

Test: Phase 1 (baseline) vs Phase 2 (shock) relative spread, per ticker

Observatation of results: 
- NVDA had the largest change of 39.1%, it also had the largest r value and smallest p-value, indicating that it was the most affected by the phase 2 events
- AAPL was the next largest change but only at 14.4%, p-value indicates that this was significant
- AMZN, PG and CAT had changes in relative spread that were considered not statistically significant, with effect size that was neglible

In [7]:
results_t1 = []

for ticker in TICKERS:
    # Extract phase 1 and phase 2 spreads for this ticker
    s1 = cross.loc[(cross["ticker"] == ticker) & (cross["phase"] == 1), "relative_spread"].dropna()
    s2 = cross.loc[(cross["ticker"] == ticker) & (cross["phase"] == 2), "relative_spread"].dropna()

    # Run the Mann-Whitney U test
   
    u_stat, p = mannwhitneyu(s1, s2, alternative="two-sided")

    # Effect size
    r = rank_biserial_r(u_stat, len(s1), len(s2))

    pct_change = (s2.median() - s1.median()) / s1.median() * 100

    results_t1.append({
        "Ticker":         ticker,
        "N Phase1":       len(s1),
        "N Phase2":       len(s2),
        "Median P1":      round(s1.median(), 4),
        "Median P2":      round(s2.median(), 4),
        "Change %":       round(pct_change, 1),
        "p-value":        round(p, 5),
        "Significance":   significance_label(p),
        "Effect r":       round(r, 3),
        "Magnitude":      interpret_effect(r),
    })

t1 = pd.DataFrame(results_t1).set_index("Ticker")
print(t1.to_string())


        N Phase1  N Phase2  Median P1  Median P2  Change %  p-value                         Significance  Effect r   Magnitude
Ticker                                                                                                                        
AAPL         184       636     0.0230     0.0263      14.4  0.00239         **Very Significant  (p<0.01)     0.147       small
NVDA         332       820     0.0151     0.0210      39.1  0.00000  ***Extremely Significant  (p<0.001)     0.280       small
AMZN         162       584     0.0189     0.0195       3.4  0.32057            Not Significant  (p≥0.05)     0.051  negligible
PG           140       529     0.1704     0.1709       0.3  0.52535            Not Significant  (p≥0.05)     0.035  negligible
CAT          226       766     0.1019     0.1046       2.6  0.59776            Not Significant  (p≥0.05)     0.023  negligible


---
## Question 2 — Was the widening asymmetric across sectors?


### Step 2a: Kruskal-Wallis test

The **Kruskal-Wallis** test is the multi-group extension of Mann-Whitney. Instead of comparing two groups, it compares K groups simultaneously and tests whether at least one group has a different distribution.

- Null Hypothesis: all five tickers have the same Phase 2 relative spread distribution
- Alternative Hypothesis: at least one ticker differs

Interpretation of Results:
-  If Kruskal-Wallis is significant, there is asymmetry present

Effect size: We use **eta-squared (η²)**, which measures what fraction of the total variance in spreads is explained by which ticker it is.
- η² = 0.01 → small
- η² = 0.06 → medium  
- η² = 0.14 → large


Observation from Results:
- At least one sector had a significantly different spread during Phase 2. 

In [ ]:
phase2 = cross[cross["phase"] == 2]

groups = [
    phase2.loc[phase2["ticker"] == t, "relative_spread"].dropna().values
    for t in TICKERS
]

# Run the Kruskal-Wallis test
h_stat, p_kw = kruskal(*groups)

# where H is the test statistic, k is the number of groups, N is total observations
k = len(TICKERS)
N = sum(len(g) for g in groups)
eta_sq = (h_stat - k + 1) / (N - k)

print(f"Kruskal-Wallis H = {h_stat:.3f}")
print(f"p-value          = {p_kw:.6f}  {significance_label(p_kw)}")
print(f"Eta-squared (η²) = {eta_sq:.4f}")

Kruskal-Wallis H = 1675.813
p-value          = 0.000000  ***Extremely Significant  (p<0.001)
Eta-squared (η²) = 0.5020


### Step 2b: Pairwise comparisons with Bonferroni correction

Kruskal-Wallis indicates presence of asymmetry but does not tell us which ticker it occurs in. 

To determine where the asymmetry lies, we run Mann-Whitney for every pair of tickers (C(5,2) = 10 pairs).

The multiple comparisons problem:
- Running more tests inflates your false positive rate.

Bonferroni correction
- Divide significance threshold by the number of tests to account for the inflated FPR.
- With 10 pairs and α = 0.05, the adjusted threshold is **α = 0.005**. Equivalently, multiply each raw p-value by 10 and compare to the original 0.05.

Note: The correction is conservative (it may miss some real differences)


Observation from results:
- After the Bonferroni Correction, 9/10 of the pairs have results that are statistically significant

In [13]:
pairs = list(combinations(TICKERS, 2))
n_comparisons = len(pairs)
bonferroni_alpha = ALPHA / n_comparisons

print(f"Number of pairs: {n_comparisons}")
print(f"Bonferroni-adjusted alpha: {ALPHA} / {n_comparisons} = {bonferroni_alpha:.4f}")
print()

pairwise_results = []

for ta, tb in pairs:
    sa = phase2.loc[phase2["ticker"] == ta, "relative_spread"].dropna()
    sb = phase2.loc[phase2["ticker"] == tb, "relative_spread"].dropna()

    u_stat, p_raw = mannwhitneyu(sa, sb, alternative="two-sided")

    # Bonferroni adjustment: multiply raw p by number of comparisons
    # Clamp at 1.0 since a probability cannot exceed 1
    p_adj = min(p_raw * n_comparisons, 1.0)

    r = rank_biserial_r(u_stat, len(sa), len(sb))

    pairwise_results.append({
        "Pair":           f"{ta} vs {tb}",
        "Med(ta)":        round(sa.median(), 4),
        "Med(tb)":        round(sb.median(), 4),
        "p (raw)":        round(p_raw, 5),
        "p (Bonferroni)": round(p_adj, 4),
        "Significant":    "Yes" if p_adj < ALPHA else "No",
        "Effect r":       round(r, 3),
        "Magnitude":      interpret_effect(r),
    })

t2b = pd.DataFrame(pairwise_results).set_index("Pair")
print(t2b.to_string())



Number of pairs: 10
Bonferroni-adjusted alpha: 0.05 / 10 = 0.0050

              Med(ta)  Med(tb)  p (raw)  p (Bonferroni) Significant  Effect r   Magnitude
Pair                                                                                     
AAPL vs NVDA   0.0263   0.0210  0.00000          0.0000         Yes    -0.186       small
AAPL vs AMZN   0.0263   0.0195  0.00000          0.0000         Yes    -0.256       small
AAPL vs PG     0.0263   0.1709  0.00000          0.0000         Yes     0.879       large
AAPL vs CAT    0.0263   0.1046  0.00000          0.0000         Yes     0.755       large
NVDA vs AMZN   0.0210   0.0195  0.04838          0.4838          No    -0.062  negligible
NVDA vs PG     0.0210   0.1709  0.00000          0.0000         Yes     0.862       large
NVDA vs CAT    0.0210   0.1046  0.00000          0.0000         Yes     0.765       large
AMZN vs PG     0.0195   0.1709  0.00000          0.0000         Yes     0.917       large
AMZN vs CAT    0.0195   0.1046  0

---
## Question 3: Which option type spread widened more during the shock

Put-Call Parity states that call and put spreads should be roughly equal for the same strike and expiry.

During a high volatility event, put spreads typically widen more than call spreads because demand for downside protection surges, overwhelming market makers' willingness to quote tight markets. 

Interpretation of results:
- A widening gap between put and call spreads is a direct measure of how asymmetric the fear was.
- A significant difference between put and call spreads during Phase 2 is evidence that the market was pricing in directional fear, not just general uncertainty.


Observation from results:
- CAT is the only ticker where call spreads were wider than put spreads, however this was deemed not statistically significant
- r > 0 indicated that put spreads tended to be wider than call spreads

In [15]:
results_t3 = []

for ticker in TICKERS:
    subset = cross[(cross["ticker"] == ticker) & (cross["phase"] == 2)]

    calls = subset.loc[subset["side"] == "call", "relative_spread"].dropna()
    puts  = subset.loc[subset["side"] == "put",  "relative_spread"].dropna()

    u_stat, p = mannwhitneyu(calls, puts, alternative="two-sided")

    # r > 0 means puts > calls (since puts is the second argument)
    r = rank_biserial_r(u_stat, len(calls), len(puts))

    results_t3.append({
        "Ticker":        ticker,
        "N Calls":       len(calls),
        "N Puts":        len(puts),
        "Median Call":   round(calls.median(), 4),
        "Median Put":    round(puts.median(), 4),
        "Put > Call":    puts.median() > calls.median(),
        "p-value":       round(p, 5),
        "Significance":  significance_label(p),
        "Effect r":      round(r, 3),
        "Magnitude":     interpret_effect(r),
    })

t3 = pd.DataFrame(results_t3).set_index("Ticker")
print(t3.to_string())


        N Calls  N Puts  Median Call  Median Put  Put > Call  p-value                         Significance  Effect r   Magnitude
Ticker                                                                                                                          
AAPL        318     318       0.0248      0.0277        True  0.02642               *Significant  (p<0.05)     0.102       small
NVDA        410     410       0.0198      0.0222        True  0.12217            Not Significant  (p≥0.05)     0.062  negligible
AMZN        292     292       0.0176      0.0216        True  0.00024  ***Extremely Significant  (p<0.001)     0.176       small
PG          264     265       0.1657      0.1782        True  0.13662            Not Significant  (p≥0.05)     0.075  negligible
CAT         383     383       0.1089      0.0968       False  0.53740            Not Significant  (p≥0.05)    -0.026  negligible


---
## Question 4 — Did spreads fully recover?

**Comparison:** Phase 1 (baseline) vs Phase 3 (recovery) relative spread, per ticker

Interpretation of Results:
- p ≥ 0.05, Phase 3 is statistically indistinguishable from Phase 1 
- p < 0.05, Spreads are still significantly elevated
- p < 0.05, Spreads fell significantly below the pre-shock baseline

Overshot recovery is common in markets after a fear event, once uncertainty resolves, liquidity improves as market makers compete more aggressively, sometimes producing tighter spreads than before the shock.

Observation from results:
- All sectors had spreads that fell significantly below the pre-shock baseline. These results were also statistically significant
- CAT was the only sector where phase 3 was statistically indistinguishable from phase 1, however these results were deemed as statistically insignificant

In [ ]:
results_t4 = []

for ticker in TICKERS:
    s1 = cross.loc[(cross["ticker"] == ticker) & (cross["phase"] == 1), "relative_spread"].dropna()
    s3 = cross.loc[(cross["ticker"] == ticker) & (cross["phase"] == 3), "relative_spread"].dropna()

    u_stat, p = mannwhitneyu(s1, s3, alternative="two-sided")
    r = rank_biserial_r(u_stat, len(s1), len(s3))

    pct_change = (s3.median() - s1.median()) / s1.median() * 100

    if p >= ALPHA:
        status = "Recovered"
    elif s3.median() > s1.median():
        status = "Not recovered"
    else:
        status = "Overshot"

    results_t4.append({
        "Ticker":       ticker,
        "Median P1":    round(s1.median(), 4),
        "Median P3":    round(s3.median(), 4),
        "Change %":     round(pct_change, 1),
        "p-value":      round(p, 5),
        "Significance": significance_label(p),
        "Effect r":     round(r, 3),
        "Magnitude":    interpret_effect(r),
        "Status":       status,
    })

t4 = pd.DataFrame(results_t4).set_index("Ticker")
print(t4.to_string())

        Median P1  Median P3  Change %  p-value                         Significance  Effect r   Magnitude     Status
Ticker                                                                                                               
AAPL       0.0230     0.0202     -12.1  0.02894               *Significant  (p<0.05)    -0.101       small   Overshot
NVDA       0.0151     0.0108     -28.3  0.00000  ***Extremely Significant  (p<0.001)    -0.289       small   Overshot
AMZN       0.0189     0.0166     -12.1  0.01609               *Significant  (p<0.05)    -0.118       small   Overshot
PG         0.1704     0.1411     -17.2  0.01886               *Significant  (p<0.05)    -0.124       small   Overshot
CAT        0.1019     0.1138      11.7  0.12855            Not Significant  (p≥0.05)     0.063  negligible  Recovered
